# Valutazione centralizzata — tutti e 4 i backbone (ricetta `acq_mild`)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

BASE = Path('/content/drive/MyDrive/2026_MLinf_gr41/Waste-Project')
DRIVE_DATASET_DIR = BASE / 'dataset'
DATASET_DIR       = Path('/content/dataset_local')
SPLIT_CSV         = BASE / 'splits' / 'split.csv'
MODELS_DIR        = BASE / 'models'
RESULTS_DIR       = BASE / 'results'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE    = 224
NUM_CLASSES = 8
SEED        = 1234
EPOCHS      = 15
RECIPE_TAG  = 'acq_mild'
SKIP_IF_EXISTS = True

CLASS_NAMES    = ['battery','clothing','glass','metal','organic','papery','plastic','undifferentiated']
TARGET_CLASSES = ['metal', 'plastic']
IMAGENET_MEAN  = [0.485, 0.456, 0.406]
IMAGENET_STD   = [0.229, 0.224, 0.225]

# Tutti i backbone del progetto
# Il sottoinsieme da allenare in questo notebook è definito da TRAIN_BACKBONES qui sotto
ALL_BACKBONES = {
    'resnet18'     : {'opt': 'sgd',   'micro': 64, 'accum': 1},
    'regnety16gf'  : {'opt': 'sgd',   'micro': 32, 'accum': 2},
    'effv2s'       : {'opt': 'sgd',   'micro': 16, 'accum': 4},
    'convnext_tiny': {'opt': 'adamw', 'micro': 32, 'accum': 2},
}

TRAIN_BACKBONES = dict(ALL_BACKBONES)

FAMILIES    = ['geometric', 'acquisition', 'background', 'resolution']
INTENSITIES = ['mild', 'moderate']

In [ ]:
import io, os, random, time, shutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image, ImageEnhance, ImageFilter
from tqdm.auto import tqdm
from sklearn.metrics import balanced_accuracy_score, recall_score
import matplotlib.pyplot as plt

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device, '|', torch.cuda.get_device_name(0) if device.type=='cuda' else 'CPU')

if not DATASET_DIR.exists():
    print('Copio il dataset in locale...')
    t0 = time.time(); shutil.copytree(DRIVE_DATASET_DIR, DATASET_DIR)
    print(f'Fatto in {time.time()-t0:.0f}s')
else:
    print('Copia locale gia presente.')

In [ ]:
df_all = pd.read_csv(SPLIT_CSV)
df_train = df_all[df_all['split'] == 'train'].reset_index(drop=True).copy()
df_val   = df_all[df_all['split'] == 'val'].reset_index(drop=True).copy()
print(f"Train: {len(df_train)}  |  Val: {len(df_val)}")

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
preprocess = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

In [ ]:
def _rng(idx, family, intensity):
    fam_id = {'geometric':1, 'acquisition':2, 'background':3, 'resolution':4}[family]
    int_id = {'mild':1, 'moderate':2}[intensity]
    return np.random.RandomState((SEED * 100003 + idx * 97 + fam_id * 13 + int_id) % (2**32))

def perturb_geometric(img, idx, intensity):
    r = _rng(idx, 'geometric', intensity)
    max_rot, min_scale = (8.0, 0.85) if intensity == 'mild' else (15.0, 0.60)
    W, H = img.size
    ang = r.uniform(-max_rot, max_rot)
    img = img.rotate(ang, resample=Image.BILINEAR, expand=False, fillcolor=(255, 255, 255))
    scale = r.uniform(min_scale, 1.0)
    cw, ch = max(1, int(W * np.sqrt(scale))), max(1, int(H * np.sqrt(scale)))
    x0 = r.randint(0, max(1, W - cw + 1)); y0 = r.randint(0, max(1, H - ch + 1))
    return img.crop((x0, y0, x0 + cw, y0 + ch)).resize((W, H), Image.BILINEAR)

def perturb_acquisition(img, idx, intensity):
    r = _rng(idx, 'acquisition', intensity)
    blur, q, jit, noise = (0.6, 70, 0.10, 4.0) if intensity == 'mild' else (1.2, 40, 0.20, 10.0)
    img = img.filter(ImageFilter.GaussianBlur(radius=blur * r.uniform(0.7, 1.3)))
    for Enh in (ImageEnhance.Brightness, ImageEnhance.Contrast, ImageEnhance.Color):
        img = Enh(img).enhance(1.0 + r.uniform(-jit, jit))
    buf = io.BytesIO(); img.save(buf, format='JPEG', quality=int(q)); buf.seek(0)
    img = Image.open(buf).convert('RGB')
    arr = np.asarray(img).astype(np.float32) + r.normal(0, noise, (img.size[1], img.size[0], 3))
    return Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))

def perturb_resolution(img, idx, intensity):
    r = _rng(idx, 'resolution', intensity)
    f = (0.5 if intensity == 'mild' else 0.34) * r.uniform(0.9, 1.1)
    W, H = img.size
    small = img.resize((max(1, int(W * f)), max(1, int(H * f))), Image.BILINEAR)
    return small.resize((W, H), Image.BILINEAR)

def perturb_background(img, idx, intensity):
    keep = 0.80 if intensity == 'mild' else 0.65
    W, H = img.size
    cw, ch = int(W * keep), int(H * keep)
    x0, y0 = (W - cw) // 2, (H - ch) // 2
    canvas = Image.new('RGB', (W, H), (255, 255, 255))
    canvas.paste(img.crop((x0, y0, x0 + cw, y0 + ch)), (x0, y0))
    return canvas

PERTURB = {'geometric': perturb_geometric, 'acquisition': perturb_acquisition,
           'background': perturb_background, 'resolution': perturb_resolution}

In [ ]:
def build_backbone(name):
    if name == 'resnet18':
        m = models.resnet18(weights='DEFAULT');           m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
    elif name == 'regnety16gf':
        m = models.regnet_y_1_6gf(weights='DEFAULT');      m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
    elif name == 'effv2s':
        m = models.efficientnet_v2_s(weights='DEFAULT');   m.classifier[1] = nn.Linear(m.classifier[1].in_features, NUM_CLASSES)
    elif name == 'convnext_tiny':
        m = models.convnext_tiny(weights='DEFAULT');       m.classifier[2] = nn.Linear(m.classifier[2].in_features, NUM_CLASSES)
    else:
        raise ValueError(name)
    return m

class TrainDataset(Dataset):
    def __init__(self, df): self.items = list(zip(df['filepath'].tolist(), df['label'].tolist()))
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        fp, lab = self.items[i]
        return train_tf(Image.open(DATASET_DIR / fp).convert('RGB')), lab

class ValDataset(Dataset):
    def __init__(self, df, family=None, intensity=None):
        self.items = list(zip(df['filepath'].tolist(), df['label'].tolist()))
        self.family, self.intensity = family, intensity
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        fp, lab = self.items[i]
        img = Image.open(DATASET_DIR / fp).convert('RGB')
        if self.family is not None:
            img = PERTURB[self.family](img, i, self.intensity)
        return preprocess(img), lab

@torch.no_grad()
def evaluate(model, df, family=None, intensity=None, bs=64):
    model.eval()
    dl = DataLoader(ValDataset(df, family, intensity), batch_size=bs, shuffle=False, num_workers=2)
    ys, ps = [], []
    for x, y in dl:
        ps.append(model(x.to(device)).argmax(1).cpu().numpy()); ys.append(np.asarray(y))
    y = np.concatenate(ys); p = np.concatenate(ps)
    bal = balanced_accuracy_score(y, p)
    tpr = recall_score(y, p, labels=list(range(NUM_CLASSES)), average=None, zero_division=0)
    return bal, dict(zip(CLASS_NAMES, tpr))

In [ ]:
@torch.no_grad()
def measure_test_memory(model, bs=32):
    if device.type != 'cuda': return float('nan')
    model.eval()
    x = torch.randn(bs, 3, IMG_SIZE, IMG_SIZE, device=device)
    torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize()
    _ = model(x); torch.cuda.synchronize()
    return torch.cuda.max_memory_allocated() / 1e9

@torch.no_grad()
def measure_fps(model, bs=32, iters=40, warmup=10):
    model.eval()
    x = torch.randn(bs, 3, IMG_SIZE, IMG_SIZE, device=device)
    for _ in range(warmup): _ = model(x)
    if device.type == 'cuda': torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(iters): _ = model(x)
    if device.type == 'cuda': torch.cuda.synchronize()
    return iters * bs / (time.time() - t0)

## Controllo: tutti e 4 i pesi devono essere presenti

In [ ]:
# La valutazione deve girare con tutti e 4 i pesi presenti
missing = [n for n in ALL_BACKBONES if not (MODELS_DIR / f"{n}_{RECIPE_TAG}.pth").exists()]
if missing:
    raise FileNotFoundError(
        f"Mancano i pesi per: {missing}.")
print("Tutti e 4 i pesi _acq_mild.pth presenti. Procedo con la valutazione su questa GPU.")

## Valutazione completa di ogni backbone

In [ ]:
rows_long, summary = [], []
for name in ALL_BACKBONES:
    path = MODELS_DIR / f"{name}_{RECIPE_TAG}.pth"
    print(f"\n=== Valutazione {name} ===")
    model = build_backbone(name).to(device)
    model.load_state_dict(torch.load(path, map_location='cpu')); model.eval()

    clean_bal, clean_tpr = evaluate(model, df_val)
    print(f"  clean-val balAcc = {clean_bal:.4f}  (metal {clean_tpr['metal']:.3f}, plastic {clean_tpr['plastic']:.3f})")

    fam_int = {}
    for fam in FAMILIES:
        for inten in INTENSITIES:
            bal, tpr = evaluate(model, df_val, fam, inten)
            fam_int[(fam, inten)] = (bal, tpr)
            rows_long.append({'backbone': name, 'family': fam, 'intensity': inten,
                              'balacc': round(bal, 4),
                              **{f'tpr_{c}': round(tpr[c], 4) for c in TARGET_CLASSES}})

    def fam_mean(inten): return float(np.mean([fam_int[(f, inten)][0] for f in FAMILIES]))
    hard_mild, hard_mod = fam_mean('mild'), fam_mean('moderate')
    hard_mean = (hard_mild + hard_mod) / 2
    acq_mean  = (fam_int[('acquisition','mild')][0] + fam_int[('acquisition','moderate')][0]) / 2

    test_mem = measure_test_memory(model)
    fps      = measure_fps(model)
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    size_mb  = os.path.getsize(path) / 1e6

    summary.append({
        'backbone': name, 'clean_val': round(clean_bal, 4),
        'hard_mild': round(hard_mild, 4), 'hard_moderate': round(hard_mod, 4),
        'hard_mean': round(hard_mean, 4), 'acquisition_mean': round(acq_mean, 4),
        'metal_tpr_acq_mod': round(fam_int[('acquisition','moderate')][1]['metal'], 4),
        'plastic_tpr_acq_mod': round(fam_int[('acquisition','moderate')][1]['plastic'], 4),
        'metal_tpr_clean': round(clean_tpr['metal'], 4), 'plastic_tpr_clean': round(clean_tpr['plastic'], 4),
        'test_mem_gb': round(test_mem, 3), 'fps': round(fps, 1),
        'params_M': round(n_params, 1), 'weights_MB': round(size_mb, 1),
    })
    print(f"  hard_mean = {hard_mean:.4f} | acq_mean = {acq_mean:.4f} | test_mem = {test_mem:.2f} GB | fps = {fps:.1f}")
    del model; torch.cuda.empty_cache()

long_df = pd.DataFrame(rows_long)
sum_df  = pd.DataFrame(summary).sort_values('hard_mean', ascending=False).reset_index(drop=True)
long_df.to_csv(RESULTS_DIR / 'backbone_acq_mild_long.csv', index=False)
sum_df.to_csv(RESULTS_DIR / 'backbone_acq_mild_summary.csv', index=False)
print('\nSalvati: backbone_acq_mild_long.csv | backbone_acq_mild_summary.csv')
sum_df

## Tabelle

In [ ]:
t98 = sum_df[['backbone', 'clean_val', 'metal_tpr_clean', 'plastic_tpr_clean', 'weights_MB']] \
        .rename(columns={'clean_val':'best_clean_val_balacc', 'metal_tpr_clean':'metal_TPR',
                         'plastic_tpr_clean':'plastic_TPR', 'weights_MB':'weights_size_MB'})
t98.to_csv(RESULTS_DIR / 'backbone_acq_mild_clean_sanity.csv', index=False)
print('=== Sanity clean-validation ===')
print(t98.to_string(index=False))

t99 = sum_df[['backbone', 'hard_mean', 'test_mem_gb', 'fps', 'metal_tpr_acq_mod', 'plastic_tpr_acq_mod']] \
        .rename(columns={'hard_mean':'hard_val_mean', 'test_mem_gb':'test_mem_GB',
                         'metal_tpr_acq_mod':'metal_TPR_shift', 'plastic_tpr_acq_mod':'plastic_TPR_shift'})
t99.to_csv(RESULTS_DIR / 'backbone_acq_mild_robust_efficiency.csv', index=False)
print('\n=== Robustezza (hard-val) + efficienza ===')
print(t99.to_string(index=False))

## Decision rule pre-registrata

In [ ]:
NOISE = 0.01
ranked = sum_df.sort_values('hard_mean', ascending=False).reset_index(drop=True)
top = ranked.iloc[0]['hard_mean']
contenders = ranked[ranked['hard_mean'] >= top - NOISE].copy()

print("Hard-val (mean mild+moderate), decrescente:")
print(ranked[['backbone','hard_mean','test_mem_gb','fps']].to_string(index=False))

if len(contenders) == 1:
    winner = contenders.iloc[0]['backbone']
    reason = f"hard_mean piu alta ({contenders.iloc[0]['hard_mean']:.4f}), oltre la soglia di rumore."
else:
    print(f"\nPareggio entro {NOISE} tra: {', '.join(contenders['backbone'])} -> tie-break su efficienza.")
    c = contenders.sort_values(['fps', 'test_mem_gb'], ascending=[False, True])
    winner = c.iloc[0]['backbone']
    reason = f"a parita' di robustezza (entro {NOISE}), efficienza migliore (fps {c.iloc[0]['fps']}, mem {c.iloc[0]['test_mem_gb']} GB)."

w = sum_df[sum_df['backbone'] == winner].iloc[0]
budget_ok = (w['test_mem_gb'] < 4.0) or np.isnan(w['test_mem_gb'])
print(f"\n>>> BACKBONE SELEZIONATO: {winner}")
print(f"    motivo: {reason}")
print(f"    test memory = {w['test_mem_gb']} GB  ({'OK < 4GB' if budget_ok else 'ATTENZIONE: supera 4GB!'})")
print(f"    pesi del modello finale: {winner}_{RECIPE_TAG}.pth")

## Grafici: robustezza e frontiera accuratezza/efficienza

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
b = sum_df.set_index('backbone')[['clean_val', 'hard_mean']].sort_values('hard_mean', ascending=False)
b.plot(kind='bar', ax=axes[0]); axes[0].set_ylim(0.80, 1.0)
axes[0].set_title('Clean-val vs Hard-val (mean) per backbone'); axes[0].set_ylabel('Balanced Accuracy')
axes[0].tick_params(axis='x', rotation=20)
ax = axes[1]
ax.scatter(sum_df['fps'], sum_df['hard_mean'], s=(sum_df['test_mem_gb'].fillna(1) * 200), alpha=0.6)
for _, r in sum_df.iterrows():
    ax.annotate(r['backbone'], (r['fps'], r['hard_mean']), fontsize=9, xytext=(5, 5), textcoords='offset points')
ax.set_xlabel('Frame rate (img/s, forward pass)'); ax.set_ylabel('Hard-val (mean)')
ax.set_title('Frontiera robustezza / velocita (dim. punto ~ memoria test)')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'backbone_acq_mild_frontier.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Confronto No-TTA vs. TTA sulla hard-val

import torch, numpy as np
import torch.nn.functional as F
from torch.utils.data import DataLoader

NOISE_FLOOR = 0.005

m = build_backbone('regnety16gf')
m.load_state_dict(torch.load(MODELS_DIR / f'regnety16gf_{RECIPE_TAG}.pth', map_location='cpu'))
m.to(device).eval()

@torch.no_grad()
def eval_balacc(model, df, family, intensity, use_tta, bs=64):
    dl = DataLoader(ValDataset(df, family, intensity), batch_size=bs, shuffle=False, num_workers=2)
    ys, ps = [], []
    for x, y in dl:
        x = x.to(device)
        if use_tta:
            both = torch.cat([x, torch.flip(x, dims=[3])], dim=0)
            probs = F.softmax(model(both), dim=1)
            B = x.shape[0]
            probs = 0.5 * (probs[:B] + probs[B:])
        else:
            probs = F.softmax(model(x), dim=1)
        ps.append(probs.argmax(1).cpu().numpy()); ys.append(np.asarray(y))
    return balanced_accuracy_score(np.concatenate(ys), np.concatenate(ps))

def hardval_mean(model, use_tta):
    scores = [eval_balacc(model, df_val, fam, inten, use_tta)
              for fam in FAMILIES for inten in INTENSITIES]
    return float(np.mean(scores))

base = hardval_mean(m, use_tta=False)
tta  = hardval_mean(m, use_tta=True)
print(f"hard-val mean  No-TTA : {base:.4f}")
print(f"hard-val mean  TTA    : {tta:.4f}")
print(f"delta = {tta - base:+.4f}   (noise floor +-{NOISE_FLOOR})")
print("Verdetto:", "guadagno reale -> la teniamo" if (tta - base) > NOISE_FLOOR
      else "dentro il rumore -> la omettiamo")